# Online Retail Sales Dashboard: From Raw Transactions to Business Insights

## Executive Summary

This portfolio project analyzes the **Online Retail II** transaction data for a UK-based, non-store online retailer. The notebook demonstrates the analytical workflow behind an executive-style sales dashboard: data loading, exploratory analysis, cleaning, feature engineering, KPI calculation, time-series analysis, product and market analysis, visualization, and automated business insights.

The final analytical output is designed to support the accompanying **Streamlit dashboard**, where users can interact with date, country, and product filters.

> **Important:** The numerical results in this notebook are calculated from the data when the notebook is run. Avoid hard-coding business conclusions so that the analysis remains reproducible if the underlying data changes.

## Business Problem

Raw transactional exports are difficult to use directly for decision-making because they may contain cancelled transactions, invalid quantities or prices, missing product descriptions, and operational/non-merchandise records. A decision-maker also needs more than charts: they need concise answers to questions such as:

- How much revenue was generated?
- How many orders and customers were served?
- Is revenue increasing or decreasing month over month?
- Which products generate the most revenue?
- Which countries contribute most to revenue?
- Where is revenue concentrated?

This project addresses those questions by transforming transaction-level data into reproducible KPIs, visualizations, and plain-language business insights. The project guide defines the target as a Python sales dashboard built with **Python, Pandas, Plotly, and Streamlit**.

## Dataset Overview

### Online Retail II

The **Online Retail II** dataset contains transactions for a UK-based, registered, non-store online retailer between **01/12/2009 and 09/12/2011**. The retailer mainly sells unique all-occasion gift-ware, and many customers are wholesalers.

**Source:** Chen, D. (2012). *Online Retail II [Dataset].* UCI Machine Learning Repository.  
https://doi.org/10.24432/C5CG6D

### Main fields used in this project

| Column | Meaning | Role in analysis |
|---|---|---|
| `Invoice` | Invoice/transaction identifier; cancellations begin with `C` | Orders and cancellation filtering |
| `StockCode` | Product code | Product analysis |
| `Description` | Product name/description | Product analysis |
| `Quantity` | Quantity purchased | Revenue calculation and validation |
| `InvoiceDate` | Transaction date/time | Time-series analysis |
| `Price` | Unit price in GBP | Revenue calculation and validation |
| `Customer ID` | Customer identifier | Customer KPI |
| `Country` | Customer country | Market analysis |

The source file used by this notebook is expected to contain two sheets: `Year 2009-2010` and `Year 2010-2011`.

## 1. Import Libraries

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load the Data

For GitHub reproducibility, the notebook expects the Excel file to live in the repository's `data/` directory while this notebook lives in `notebooks/`:

```text
sales-dashboard/
├── data/
│   └── online_retail_II.xlsx
└── notebooks/
    └── Sales dashboard project.ipynb
```

The code below also checks the current working directory so the notebook is easier to run from different environments.

In [ ]:
# Locate the dataset without using a machine-specific absolute path
candidate_paths = [
    Path("../data/online_retail_II.xlsx"),
    Path("data/online_retail_II.xlsx"),
    Path("online_retail_II.xlsx"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Place 'online_retail_II.xlsx' in the repository's "
        "data/ folder, or update candidate_paths above."
    )

print(f"Using dataset: {data_path.resolve()}")

sheet_2009_2010 = pd.read_excel(data_path, sheet_name="Year 2009-2010")
sheet_2010_2011 = pd.read_excel(data_path, sheet_name="Year 2010-2011")

df = pd.concat([sheet_2009_2010, sheet_2010_2011], ignore_index=True)

print(f"Combined dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

## 3. Initial Data Exploration

The first pass checks the dataset structure, data types, missing values, and basic cardinality before defining cleaning rules.

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing_summary = (
    df.isna().sum()
    .to_frame("missing_rows")
    .assign(missing_pct=lambda x: x["missing_rows"] / len(df) * 100)
    .sort_values("missing_rows", ascending=False)
)
display(missing_summary)

print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Key cardinality and validity checks
cancellations = df["Invoice"].astype(str).str.startswith("C")

quality_checks = pd.Series({
    "Cancelled rows": cancellations.sum(),
    "Non-positive quantity rows": (df["Quantity"] <= 0).sum(),
    "Non-positive price rows": (df["Price"] <= 0).sum(),
    "Missing descriptions": df["Description"].isna().sum(),
    "Unique customers": df["Customer ID"].nunique(dropna=True),
    "Unique countries": df["Country"].nunique(dropna=True),
    "Unique stock codes": df["StockCode"].nunique(dropna=True),
})
display(quality_checks.to_frame("count"))

## 4. Data Cleaning

### Cleaning rules

The project guide requires cancelled transactions and invalid quantities to be removed. The exploratory checks above support the following additional rules:

1. **Remove cancelled invoices** — invoices beginning with `C` represent cancellations.
2. **Remove non-positive quantities** — these do not represent normal positive sales quantities for this dashboard.
3. **Remove non-positive prices** — revenue cannot be calculated as a valid sale when the unit price is zero or negative.
4. **Remove rows with missing descriptions** — these rows are not suitable for product-level analysis and may represent operational records.
5. **Keep transactions with missing Customer ID** for transaction/revenue analysis, but exclude missing IDs from the unique-customer KPI. This avoids silently removing otherwise valid sales from revenue totals.

The notebook does **not** remove the non-merchandise stock codes from overall revenue. Those codes are excluded specifically from the **product ranking**, where the business question is which merchandise products generate the most revenue.

In [ ]:
# Explore non-cancelled rows with invalid quantities
not_cancelled = ~df["Invoice"].astype(str).str.startswith("C")
bad_qty_not_cancelled = df[(df["Quantity"] <= 0) & not_cancelled]

print(f"Non-cancelled rows with non-positive quantity: {len(bad_qty_not_cancelled):,}")
display(
    bad_qty_not_cancelled[
        ["Invoice", "StockCode", "Description", "Quantity", "Price"]
    ].head(10)
)

In [ ]:
# Inspect non-positive prices and their descriptions
price_zero = df[df["Price"] <= 0]
print(f"Rows with non-positive price: {len(price_zero):,}")
print("Description missing status:")
display(price_zero["Description"].isna().value_counts())

display(
    df[(df["Price"] <= 0) & df["Description"].notna()]["Description"]
    .value_counts()
    .head(15)
    .to_frame("rows")
)

In [ ]:
# Apply the documented cleaning rules
df_clean = df[
    (~df["Invoice"].astype(str).str.startswith("C")) &
    (df["Description"].notna()) &
    (df["Price"] > 0) &
    (df["Quantity"] > 0)
].copy()

# Convert InvoiceDate explicitly to datetime as required by the project guide
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"], errors="coerce")

# Remove any rows where the date could not be parsed
df_clean = df_clean.dropna(subset=["InvoiceDate"]).copy()

cleaning_summary = pd.DataFrame({
    "Rows": [len(df), len(df_clean), len(df) - len(df_clean)],
}, index=["Original", "Cleaned", "Removed"])
cleaning_summary["Percent of original"] = cleaning_summary["Rows"] / len(df) * 100

display(cleaning_summary)
print(f"Rows retained: {len(df_clean) / len(df):.1%}")

### Data-cleaning result

The table above is the source of truth for how many records were removed. This is preferable to hard-coding a percentage in the narrative because the notebook remains reproducible if the source file is updated.

## 5. Feature Engineering

Revenue is calculated at transaction-line level using:

**Revenue = Quantity × Price**

This creates the main measure used throughout the dashboard.

In [ ]:
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["Price"]

# Useful date fields for grouping and interpretation
df_clean["Year"] = df_clean["InvoiceDate"].dt.year
df_clean["Month"] = df_clean["InvoiceDate"].dt.to_period("M").astype(str)

print("Revenue statistics:")
display(df_clean["Revenue"].describe().to_frame("Revenue"))

display(df_clean.head())

## 6. KPI Calculations

The project guide specifies the following main KPIs:

- Total Revenue
- Total Orders
- Average Order Value (AOV)
- Unique Customers
- Monthly Growth %
- Best-Selling Product
- Top Market

In [ ]:
# Core KPIs
total_revenue = df_clean["Revenue"].sum()
total_orders = df_clean["Invoice"].nunique()
aov = total_revenue / total_orders if total_orders else 0
unique_customers = df_clean["Customer ID"].nunique(dropna=True)

kpi_summary = pd.Series({
    "Total Revenue": total_revenue,
    "Total Orders": total_orders,
    "Average Order Value": aov,
    "Unique Customers": unique_customers,
})

display(kpi_summary.to_frame("Value"))

print(f"Total Revenue: £{total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Average Order Value: £{aov:,.2f}")
print(f"Unique Customers: {unique_customers:,}")

## 7. Monthly Revenue and Month-over-Month Growth

Monthly revenue shows the overall sales trend, while month-over-month (MoM) growth measures the percentage change from one month to the previous month.

The first month has no MoM comparison, so its growth value is naturally `NaN`.

In [ ]:
monthly_revenue = (
    df_clean.set_index("InvoiceDate")
    .resample("ME")["Revenue"]
    .sum()
    .rename("Revenue")
)

monthly_analysis = monthly_revenue.to_frame()
monthly_analysis["MoM Growth %"] = monthly_revenue.pct_change() * 100

# Latest month with a valid MoM comparison
valid_mom = monthly_analysis.dropna(subset=["MoM Growth %"])
latest_mom = valid_mom.iloc[-1] if not valid_mom.empty else None

print("Monthly revenue and MoM growth:")
display(monthly_analysis.tail(12))

if latest_mom is not None:
    print(
        f"Latest MoM growth: {latest_mom['MoM Growth %']:.1f}% "
        f"({valid_mom.index[-1].strftime('%B %Y')})"
    )

## 8. Product Analysis

For the product ranking, operational/non-merchandise stock codes are excluded so that the result answers the question **which merchandise products generate the most revenue?**

This exclusion applies to the product ranking only; it does not retroactively remove those records from the overall transaction/revenue KPI.

In [ ]:
non_product_codes = [
    "POST", "DOT", "M", "C2", "BANK CHARGES", "CRUK",
    "AMAZONFEE", "PADS", "DCGS0076"
]

product_sales = df_clean[~df_clean["StockCode"].astype(str).isin(non_product_codes)].copy()

top_products = (
    product_sales.groupby("StockCode")
    .agg(
        total_revenue=("Revenue", "sum"),
        units_sold=("Quantity", "sum"),
        description=("Description", "first"),
    )
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

display(top_products)

best_product = top_products.iloc[0] if not top_products.empty else None
if best_product is not None:
    print(
        f"Best-selling product by revenue: {best_product['description']} "
        f"(£{best_product['total_revenue']:,.2f})"
    )

## 9. Country / Market Analysis

Revenue is grouped by country to identify the largest markets and measure geographic concentration.

In [ ]:
country_revenue = (
    df_clean.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

top_countries = country_revenue.head(10)
top_market = country_revenue.index[0] if not country_revenue.empty else None
top_market_revenue = country_revenue.iloc[0] if not country_revenue.empty else 0
top_market_share = (top_market_revenue / total_revenue * 100) if total_revenue else 0

display(top_countries.to_frame("Revenue"))

if top_market is not None:
    print(f"Top market: {top_market}")
    print(f"Top-market revenue share: {top_market_share:.1f}%")

## 10. Visualizations

The project guide calls for line, bar, and donut charts. These charts are designed to answer three different business questions:

1. **Line chart:** How has revenue changed over time?
2. **Bar chart:** Which products generate the most revenue?
3. **Donut chart:** How concentrated is revenue across markets?

In [ ]:
# Line chart: monthly revenue trend
monthly_revenue_df = monthly_revenue.reset_index()
monthly_revenue_df.columns = ["Month", "Revenue"]

fig_line = px.line(
    monthly_revenue_df,
    x="Month",
    y="Revenue",
    title="Monthly Revenue Trend",
    markers=True,
    labels={"Revenue": "Revenue (£)", "Month": "Month"},
)
fig_line.update_layout(hovermode="x unified")
fig_line.show()

In [ ]:
# Bar chart: top 10 products by revenue
fig_bar_products = px.bar(
    top_products.reset_index(),
    x="total_revenue",
    y="description",
    orientation="h",
    title="Top 10 Products by Revenue",
    labels={"total_revenue": "Revenue (£)", "description": "Product"},
)
fig_bar_products.update_layout(yaxis={"categoryorder": "total ascending"})
fig_bar_products.show()

In [ ]:
# Donut chart: revenue share by country (top 5 + Other)
top5 = country_revenue.head(5)
other_total = country_revenue.iloc[5:].sum()
donut_data = pd.concat([top5, pd.Series({"Other": other_total})])

donut_df = donut_data.reset_index()
donut_df.columns = ["Country", "Revenue"]

fig_donut = px.pie(
    donut_df,
    values="Revenue",
    names="Country",
    title="Revenue Share by Country (Top 5 + Other)",
    hole=0.5,
)
fig_donut.show()

## 11. Automated Business Insights

A portfolio dashboard should turn calculations into statements that a non-technical stakeholder can understand. The project guide specifically recommends automated sentences such as revenue growth, country contribution, and the highest-revenue product.

The code below generates these statements from the calculated values rather than hard-coding the wording.

In [ ]:
insights = []

# Latest month-over-month movement
if latest_mom is not None:
    latest_growth = latest_mom["MoM Growth %"]
    direction = "increased" if latest_growth >= 0 else "decreased"
    insights.append(
        f"Revenue {direction} by {abs(latest_growth):.1f}% compared with the previous month."
    )

# Top market contribution
if top_market is not None:
    insights.append(
        f"{top_market} contributed {top_market_share:.1f}% of total revenue, "
        "indicating a concentrated geographic revenue base."
    )

# Best product
if best_product is not None:
    insights.append(
        f"{best_product['description']} generated the highest product revenue "
        f"at £{best_product['total_revenue']:,.2f}."
    )

# Highest-revenue month
if not monthly_revenue.empty:
    best_month = monthly_revenue.idxmax()
    best_month_revenue = monthly_revenue.max()
    insights.append(
        f"The highest-revenue month was {best_month.strftime('%B %Y')}, "
        f"with £{best_month_revenue:,.2f} in revenue."
    )

print("Business Insights")
for i, insight in enumerate(insights, start=1):
    print(f"{i}. {insight}")

## 12. Business Recommendations

Recommendations should follow from the evidence above rather than from generic retail advice. The exact wording below is generated from the analysis context and should be revisited if the dataset or cleaning rules change.

- **Monitor geographic concentration:** If one market contributes a very large share of revenue, track that market separately and consider whether growth in other markets can reduce concentration risk.
- **Use top-product performance for inventory planning:** Products with consistently high revenue contribution deserve closer monitoring of availability and replenishment.
- **Plan around seasonal demand:** Inspect the monthly trend for recurring peaks before making inventory, marketing, or staffing decisions.
- **Use MoM growth as an early-warning metric:** Large positive or negative changes should be investigated alongside order volume, customer activity, and product mix rather than interpreted in isolation.
- **Keep the dashboard interactive:** Date, country, and product filters allow stakeholders to move from the overall picture to a specific market or product without rebuilding the analysis.

## 13. Conclusion

This notebook converts raw transaction records into a reproducible sales-analysis workflow. It demonstrates data cleaning, feature engineering, KPI calculation, time-series analysis, product and country analysis, visualization, and automated business commentary.

The notebook is the analytical foundation for the accompanying Streamlit application. The dashboard should use the same cleaning rules and KPI definitions so that the interactive results remain consistent with this analysis.

## 14. Streamlit Dashboard

The completed Streamlit application is maintained separately as `sales_dashboard_final.py`. The notebook should document the application without relying on machine-specific localhost or network URLs.

For the GitHub repository, use a structure such as:

```text
sales-dashboard/
├── app.py
├── requirements.txt
├── README.md
├── data/
│   └── online_retail_II.xlsx
├── notebooks/
│   └── Sales dashboard project.ipynb
└── assets/
    └── dashboard.png
```

After deploying through Streamlit Community Cloud, add the permanent public app URL to the README rather than storing a temporary local/network URL in this notebook.

## 15. Skills Demonstrated

- Python
- Pandas
- Data cleaning and validation
- Feature engineering
- KPI development
- Time-series analysis
- Grouping and aggregation
- Plotly visualization
- Business insight generation
- Streamlit dashboard development
- Git/GitHub project organization

## 16. Next Steps

1. Commit the cleaned notebook and dashboard application to GitHub.
2. Add `requirements.txt` with the packages required to reproduce the project.
3. Add a dashboard screenshot to `assets/`.
4. Create a README containing the executive summary, business problem, methodology, skills, results/recommendations, and next steps.
5. Deploy the Streamlit application and add the permanent application URL to the README.
6. Keep the notebook and dashboard KPI definitions synchronized when either is updated.